# Aula 9: Playwright e páginas dinâmicas

Na Aula 8 a coleta era HTML estático: `requests` + BeautifulSoup. Hoje o caso muda: páginas em que o conteúdo **só aparece depois** que o JavaScript roda no navegador (botão que carrega mais itens, feed que monta cards depois, etc.).

A ferramenta de hoje é o **Playwright**: abrir um Chromium de verdade, esperar, clicar, ler o HTML atualizado.

**Site da aula:** [Ground News – Russia](https://ground.news/interest/russia)

**A prática roda no terminal**, não neste Jupyter. O roteiro de copiar e colar está em `comandos.md`. Este notebook explica o porquê e o mapa do que vamos coletar.


## 1. Revisão rápida

Da Aula 8 a gente reaproveita: DOM, `.find()` / `.find_all()` com dicionário (`{"class": "post"}`, `{"data-testid": "story-item"}`), `requests` + `status_code`, e CSV no final. Se algo disso estiver vago, vale espiar a Aula 8 antes.


## 2. Por que `requests` não basta

Em muitos sites o servidor manda um HTML inicial, e o resto (mais cards, detalhes) só entra depois de cliques e JavaScript.

`requests.get(url)` **não** abre navegador e **não** executa JavaScript. Ele vê o HTML da resposta HTTP e para.

No Ground News, o botão **More stories** é o exemplo perfeito: só depois do clique (no navegador) a lista de notícias cresce. O script `exemplos/01_requests_vs_playwright.py` mostra isso lado a lado.


## 3. O que vamos coletar

URL: `https://ground.news/interest/russia`

Fluxo (eficiente):

1. Abrir a listagem
2. Clicar **10 vezes** em More stories
3. Guardar título + URL de cada card em `dados/ground-russia-lista.csv`
4. **Depois** abrir cada `/article/...` e extrair o detalhe em `dados/ground-russia-detalhes.csv`

Em cada story:

- `summary` (resumo)
- `source_urls` (links das fontes, separados por `|`)
- painel **Coverage Details**: Total News Sources, Leaning Left, Leaning Right, Center, Last Updated, Bias Distribution

**OBS:** factuality / ownership costumam ser premium. Não coletamos isso.

**OBS 2:** os scripts usam `headless=False`, então a janela do Chromium abre e a turma vê o que acontece.


## 4. Instalação (dois passos)

1. Instalar a biblioteca Python (`playwright` no `requirements.txt`).
2. Instalar o navegador que ela controla (`playwright install chromium`). São passos **separados**.

Use o `.venv` da **raiz** do repositório da disciplina (não crie outro `.venv` dentro da pasta da aula).

### Com `uv` (preferido)

**Windows:**
```cmd
cd caminho\\para\\repositorio-da-disciplina
uv pip install -r requirements.txt
uv run playwright install chromium
```

**Mac:**
```bash
cd caminho/para/repositorio-da-disciplina
uv pip install -r requirements.txt
uv run playwright install chromium
```

### Com `.venv` ativado (`venv` / virtualenv, sem `uv`)

**Windows:**
```cmd
cd caminho\\para\\repositorio-da-disciplina
.venv\\Scripts\\activate
pip install -r requirements.txt
playwright install chromium
```

**Mac:**
```bash
cd caminho/para/repositorio-da-disciplina
source .venv/bin/activate
pip install -r requirements.txt
playwright install chromium
```

Se `playwright` não for reconhecido com o ambiente ativado: `python -m playwright install chromium`.

**OBS:** sem o `install chromium`, o código quebra no `chromium.launch()` dizendo que o executável não foi encontrado. Detalhes em `comandos.md` (e, se o `uv` falhar no lab, `tutoriais/uv_nao_funcionando.md`).


## 5. Playwright sync e Jupyter não combinam

A API síncrona (`from playwright.sync_api import sync_playwright`) **não roda** dentro de célula Jupyter. Erro típico:

```text
Error: It looks like you are using Playwright Sync API inside the asyncio loop.
```

Por isso os exemplos são scripts `.py` em `exemplos/`, rodados no **terminal**:

```bash
uv run exemplos/01_requests_vs_playwright.py
uv run exemplos/02_listar_com_more_stories.py
uv run exemplos/03_detalhar_stories.py
```

Com `.venv` ativado (sem `uv`), troque por `python exemplos/01_requests_vs_playwright.py` (e o mesmo para 02 e 03).

Siga a ordem em **`comandos.md`**. Este notebook **não** dispara a coleta: a prática é no terminal.


## 6. A API síncrona (para ler, não para rodar aqui)

```python
from playwright.sync_api import sync_playwright

with sync_playwright() as p:
    browser = p.chromium.launch(headless=False)  # False = janela visível
    context = browser.new_context(user_agent="...")  # User-Agent de navegador real
    page = context.new_page()
    page.goto("https://ground.news/interest/russia")
    page.wait_for_selector('[data-testid="story-item"]')
    cards = page.query_selector_all('[data-testid="story-item"]')
    browser.close()
```

- `with sync_playwright()` fecha tudo ao final, mesmo se der erro no meio.
- `headless=False` abre a janela (padrão desta aula).
- `wait_for_selector` espera o elemento aparecer de verdade (melhor que `time.sleep` fixo).
- `query_selector_all` / `click` / `text_content` / `get_attribute` fazem o trabalho de achar e ler.


## 7. BeautifulSoup (Aula 8) × Playwright (hoje)

| O que você quer | BeautifulSoup (Aula 8) | Playwright (hoje) |
|---|---|---|
| Achar vários | `find_all("div", {"data-testid": "story-item"})` | `query_selector_all('[data-testid="story-item"]')` |
| Achar um | `find("div", {"data-testid": "story-item"})` | `query_selector('[data-testid="story-item"]')` |
| Texto | `.get_text(strip=True)` | `.text_content().strip()` |
| Atributo | `elemento["href"]` | `.get_attribute("href")` |
| Clicar / esperar | (não tem) | `.click()` / `wait_for_selector` |

Seletores estáveis desta aula:

- `[data-testid="story-item"]` (card)
- `[data-testid="load-more-stories-button"]` (More stories)
- `h4` no card (título na lista)
- `h1` na story (título)
- texto do painel: `Coverage Details`


## 8. Os três scripts

| Script | Faz | Saída |
|---|---|---|
| `01_requests_vs_playwright.py` | HTML bruto vs 1 clique em More stories | print no terminal (+ janela) |
| `02_listar_com_more_stories.py` | 10 cliques + lista de títulos/URLs | `dados/ground-russia-lista.csv` |
| `03_detalhar_stories.py` | abre cada URL da lista | `dados/ground-russia-detalhes.csv` |

Abra os `.py`: quase toda linha está comentada. Depois rode no terminal, na ordem do `comandos.md`.

O script 03 pode demorar (muitas stories + 1s de pausa entre páginas). Nem toda story tem o painel Coverage Details completo; nesses casos as colunas ficam vazias.


## 9. Espera explícita e load more

Evite depender só de `time.sleep(2)`. Prefira:

- `page.wait_for_selector('[data-testid="story-item"]')`
- depois de clicar em More stories, esperar a **contagem** de cards crescer (`wait_for_function`)
- `scroll_into_view_if_needed()` antes do clique (senão o botão pode estar fora da tela)

Entre cliques e entre stories, os scripts ainda usam uma pausa curta (`time.sleep(1)`): uso educacional, sem martelar o site.


## 10. Fechar o navegador

Todo script usa `with sync_playwright() as p:` e `browser.close()`. O `with` garante o fechamento mesmo se uma linha no meio der erro (mesma lógica do `with open(...)`).


## 11. Ética

- `robots.txt` do Ground News: `Allow: /` (exceto `/mediaopoly`).
- Pausa entre páginas; material educacional.
- Não contornar login/paywall; não martelar o site.
- Classes CSS mudam fácil; por isso preferimos `data-testid` e o texto "Coverage Details".


## 12. Quando der errado

- **`TimeoutError` esperando seletor:** seletor errado, cookie na frente, rede lenta, ou o site mudou. Feche cookies (`Accept`), confira o `data-testid`, aumente o `timeout`.
- **Executável do navegador não encontrado:** falta `playwright install chromium` (Seção 4).
- **Sync API inside the asyncio loop:** tentou Playwright sync numa célula Jupyter. Use o terminal.
- **`403` no `requests`:** use `User-Agent` de navegador (o script 01 já faz).
- **Lista não cresce no More stories:** `scroll_into_view_if_needed()` antes do clique; rode de novo se a rede falhar.
- **`ModuleNotFoundError: playwright`:** ambiente/deps. Volte na Seção 4.


## Prática

1. Siga `comandos.md` no terminal (`01` → `02` → `03`).
2. Depois abra `exercicios/exercicio-09-playwright-paginas-dinamicas.ipynb`: outro interesse do Ground News (sugestão: sanctions), **2** cliques em More stories, script `.py` próprio no terminal.
